<a href="https://colab.research.google.com/github/ascr0wn/google_colab_notebooks/blob/main/01_dl_intro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
'''
Douts/questions:
1. what exactly does numpy and pytorch freezes when we set a manual seed in them in our pipeline!?
2. explain: "the vectorized line is dramatically faster, typically by tens of times. NumPy and PyTorch run whole-array operations in optimized compiled code, which is why a training loop works on batches and not on one number at a time"
3.
'''

'\nDouts/questions:\n1. what exactly does numpy and pytorch freezes when we set a manual seed in them in our pipeline!?\n2. explain: "the vectorized line is dramatically faster, typically by tens of times. NumPy and PyTorch run whole-array operations in optimized compiled code, which is why a training loop works on batches and not on one number at a time"\n3.\n'

In [2]:
### 0.1 - Setup ###

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

print("Torch Version: ", torch.__version__)
print("Numpy version: ", np.__version__)
print("GPU Available: ", torch.cuda.is_available())

Torch Version:  2.11.0+cu130
Numpy version:  2.1.3
GPU Available:  True


In [4]:
import random

SEED = 7
np.random.seed(SEED)
torch.manual_seed(SEED) #weight initialization, dropout, data shuffling, random augmentation, random sampling

if torch.cuda.is_available():
  torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using Device: ", device)

Using Device:  cuda


In [5]:
x = torch.rand(3,4) #pytorch tensor 3 rows, 4 columns, filled with random values [0,1)
print("x", x)
print("x's shape: ", x.shape)
print("x's device: ", x.device) #by default it is allotted on a cpu
x = x.to(device) #change allocation to device(here a cuda gpu)
print("x's new device: ", x.device)

x tensor([[0.5349, 0.1988, 0.6592, 0.6569],
        [0.2328, 0.4251, 0.2071, 0.6297],
        [0.3653, 0.8513, 0.8549, 0.5509]])
x's shape:  torch.Size([3, 4])
x's device:  cpu
x's new device:  cuda:0


In [6]:
### 0.2 - NUMPY ###

In [7]:
a = np.array([1, 2, 3])
b = np.array([[1,2,3],[4,5,6]])
c = np.zeros((3,3))
d = np.random.rand(3,2)

print(a.shape, a.dtype, a.ndim)
print(b.shape, b.dtype, b.ndim)
print(c.shape, c.dtype, c.ndim)
print(d.shape, d.dtype, d.ndim)

(3,) int64 1
(2, 3) int64 2
(3, 3) float64 2
(3, 2) float64 2


In [8]:
e = np.arange(12)
e = e.reshape(3,4)
print(e.shape, e.dtype, e.ndim, "\n")
print(e, "\n")
print(e[1,:], "\n")
print(e[:,3], "\n")
print(e[2,2], "\n")
print(e[1:,:2], "\n")
print(e.reshape(-1,2).shape) #find number of rows on your own, but number of columns are 2, eg: can we used to flatten an image (28x28 -> 784)

(3, 4) int64 2 

[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]] 

[4 5 6 7] 

[ 3  7 11] 

10 

[[4 5]
 [8 9]] 

(6, 2)


In [9]:
f = np.random.rand(1_000_000)
%timeit sum(v*2 for v in f) # using loops is slow
%timeit (f*2).sum() # numpy (compiled optimised code) is faster

173 ms ± 2.42 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
1.02 ms ± 17.4 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)


In [20]:
g = np.arange(12).reshape(3,4) # (3,4) matrix with values [0-11]
print(g.sum())
print(g.sum(axis=0)) #sums columns wise, drops row axis so shape becomes (4,)
print(g.sum(axis=1)) #sums row wise, drops column axis so shape becomes (3,)
print(g.max(axis=0)) #max in every column, shape (4,)
print(g.argmax(axis=1)) #index of max in every row (3,)

66
[12 15 18 21]
[ 6 22 38]
[ 8  9 10 11]
[3 3 3]


In [11]:
h = np.array([[1,2,3], [4,5,6]]) # shape: 2x3
i = np.array([1,2]) # shape: 2,
j = np.array([[1], [2]]) # shape: 2x1
k = np.array([1,2,3]) #shape: 3,
print(h+j, "\n", h+k) #h+i is bad, doesn't match!
# broadcast!, mostly used for adding bias. stretches dimention if not matched in special conditions only.
# start from right dimention (column), it's good if dimention matches or one of them is 1 (stretch), else not good!
#eg: 2x3 with 2,: bad (2 not match with 3), 2x3 with 2x1: good(3 with 1 and 2 with 2), 2x3 with 3,:good (3 matches with 3)

[[2 3 4]
 [6 7 8]] 
 [[2 4 6]
 [5 7 9]]


In [12]:
'''
3 features/inputs, 2 neurons in hidden layer, if batch size is 4 then input is 4x3 matrix, weight matrix would be
3x2, bias is 2, then final input to neurons is a 4x2 matrix where rows are sample number and columns decide neuron
number. for example row 3 column 2 is 3rd sample for neuron 2.
here bias is broadcasted to all rows of matrix 4x2.
finally 4x2 goes into activation function, let's say relu and we get activated 4x2 output.
'''

"\n3 features/inputs, 2 neurons in hidden layer, if batch size is 4 then input is 4x3 matrix, weight matrix would be\n3x2, bias is 2, then final input to neurons is a 4x2 matrix where rows are sample number and columns decide neuron\nnumber. for example row 3 column 2 is 3rd sample for neuron 2.\nhere bias is broadcasted to all rows of matrix 4x2.\nfinally 4x2 goes into activation function, let's say relu and we get activated 4x2 output.\n"

In [13]:
l = np.array([[1, 2, 3],
             [2, 3, 1],
             [3, 1, 2]])

m = np.array([[1, 2],
             [2, 1],
             [2, 3]])

print(l@m) # this is matrix multiplication
#print(l*m) # this is overlap multiplication between matrices, element by element (can be broadcasted, if dimentions don't match)


[[11 13]
 [10 10]
 [ 9 13]]


In [14]:
#Manual feedforward
np.random.seed(42)
X = np.random.rand(4,3)
W = np.random.rand(3,2)
b = np.random.rand(2,)
Z = X @ W + b
H = np.maximum(0,Z)
print("input: ", X, "shape: ", X.shape)
print("weights: ", W, "shape: ", W.shape)
print("bias: ", b, "shape: ", b.shape)
print("pre-activated output: ", Z, "shape: ", Z.shape)
print("post-activation: ", H, "shape: ", H.shape)

input:  [[0.37454012 0.95071431 0.73199394]
 [0.59865848 0.15601864 0.15599452]
 [0.05808361 0.86617615 0.60111501]
 [0.70807258 0.02058449 0.96990985]] shape:  (4, 3)
weights:  [[0.83244264 0.21233911]
 [0.18182497 0.18340451]
 [0.30424224 0.52475643]] shape:  (3, 2)
bias:  [0.43194502 0.29122914] shape:  (2,)
pre-activated output:  [[1.13929526 0.92924248]
 [1.00612208 0.5288214 ]
 [0.82067332 0.77786214]
 [1.32020515 0.95432236]] shape:  (4, 2)
post-activation:  [[1.13929526 0.92924248]
 [1.00612208 0.5288214 ]
 [0.82067332 0.77786214]
 [1.32020515 0.95432236]] shape:  (4, 2)


In [15]:
X = np.random.rand(100, 3) * 50 + 10
X_normalised = (X - X.mean(axis=0)) / X.std(axis=0)
print(X_normalised)

[[ 0.38999944 -1.14726718 -0.76149018]
 [-0.4556857  -0.04340413  0.88001601]
 [-1.02990465  0.15940836  0.2382332 ]
 [-1.55773948  0.4847706  -1.16641546]
 [-1.4936608   1.67498472  1.48082929]
 [ 1.06706983 -0.57151383 -1.40897009]
 [ 0.63933969 -0.09890652 -1.32784514]
 [-0.01193434 -1.51375692  1.29334446]
 [-0.82629139  0.67647033 -0.69634541]
 [ 0.07381234  0.27264773 -1.11870383]
 [ 1.6223393   1.0691298   1.39382136]
 [ 1.36481008  0.45114008  1.33514139]
 [-1.41290988 -0.9502968  -1.58358071]
 [-0.59703399 -0.27839439 -0.83072743]
 [ 1.13713907 -0.38970943 -0.79881339]
 [ 0.15176311 -1.14227967  0.93668606]
 [-1.46093779  1.80749105  0.8369627 ]
 [-1.03320519 -1.61441062  0.98084892]
 [ 0.71727756  0.90829516  0.83371844]
 [-1.46268087 -0.38373849 -1.34838489]
 [ 1.25552524  0.53970039 -0.63246408]
 [-1.49880483 -0.54930745 -0.6514907 ]
 [ 0.79564437  0.58942101  1.21973887]
 [-0.09103543 -1.21665476  0.64052723]
 [ 0.90305157  0.32344083  0.83270907]
 [-0.01669279  0.18904123

In [ ]:
### --- pandas (loads and inspect data) and matplotlib (draws data) --- ###

In [25]:
# We use Pandas DataFrames because they make data cleaning, filtering and column selection easier;
# we use NumPy arrays because they are better suited for fast numerical operations and matrix calculations.
# Basic difference: DataFrames have named rows and columns, whereas NumPy arrays mainly work with positional indices and don't inherently store column names.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
#Pandas: Explore, clean, select features, handle missing values, Numpy: Convert, normalize, split and transform
raw = load_breast_cancer(as_frame=True).frame
#original returns bunch object (data.data,.target,.feature_names,.target_names,.DESCR - they are all numpy ndarray, .DESCR is string
#as_frame=True returns still a bunch object but data.data(shape: axb),.target(shape: a,)(it is a series, not a dataframe),.frame(shape: axb+1) as pandas dataframe
#.frame changes it a dataframe complete which is data+target. no bunch object just 1 combined dataframe!
raw.to_csv("cancer.csv", index=False)
df = pd.read_csv("cancer.csv")
print("data shape: ", df.shape)
df.head()

data shape:  (569, 31)


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0


In [29]:
df.info() #number of rows and columns, column names, non-null values count/column, data types/column, memory usage
print(df.isna().sum().sum()) #flow-missing values per column summed to find total.

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 31 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   mean radius              569 non-null    float64
 1   mean texture             569 non-null    float64
 2   mean perimeter           569 non-null    float64
 3   mean area                569 non-null    float64
 4   mean smoothness          569 non-null    float64
 5   mean compactness         569 non-null    float64
 6   mean concavity           569 non-null    float64
 7   mean concave points      569 non-null    float64
 8   mean symmetry            569 non-null    float64
 9   mean fractal dimension   569 non-null    float64
 10  radius error             569 non-null    float64
 11  texture error            569 non-null    float64
 12  perimeter error          569 non-null    float64
 13  area error               569 non-null    float64
 14  smoothness error         5

In [28]:
df.describe() # column wise summaries like count(non missing values), mean, std, min, 25%, 50%, 75%, max(value)

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
count,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,...,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000
mean,14.127292,19.289649,91.969033,654.889104,0.096360,0.104341,0.088799,0.048919,0.181162,0.062798,...,25.677223,107.261213,880.583128,0.132369,0.254265,0.272188,0.114606,0.290076,0.083946,0.627417
std,3.524049,4.301036,24.298981,351.914129,0.014064,0.052813,0.079720,0.038803,0.027414,0.007060,...,6.146258,33.602542,569.356993,0.022832,0.157336,0.208624,0.065732,0.061867,0.018061,0.483918
min,6.981000,9.710000,43.790000,143.500000,0.052630,0.019380,0.000000,0.000000,0.106000,0.049960,...,12.020000,50.410000,185.200000,0.071170,0.027290,0.000000,0.000000,0.156500,0.055040,0.000000
25%,11.700000,16.170000,75.170000,420.300000,0.086370,0.064920,0.029560,0.020310,0.161900,0.057700,...,21.080000,84.110000,515.300000,0.116600,0.147200,0.114500,0.064930,0.250400,0.071460,0.000000
50%,13.370000,18.840000,86.240000,551.100000,0.095870,0.092630,0.061540,0.033500,0.179200,0.061540,...,25.410000,97.660000,686.500000,0.131300,0.211900,0.226700,0.099930,0.282200,0.080040,1.000000
75%,15.780000,21.800000,104.100000,782.700000,0.105300,0.130400,0.130700,0.074000,0.195700,0.066120,...,29.720000,125.400000,1084.000000,0.146000,0.339100,0.382900,0.161400,0.317900,0.092080,1.000000
max,28.110000,39.280000,188.500000,2501.000000,0.163400,0.345400,0.426800,0.201200,0.304000,0.097440,...,49.540000,251.200000,4254.000000,0.222600,1.058000,1.252000,0.291000,0.663800,0.207500,1.000000


In [39]:
s = df["mean radius"] #series not a dataframe
print(s.shape)
d = df[["mean radius", "mean area"]] #d is a dataframe
print(d.shape)
print(df.iloc[:4,:4]) #iloc[row_position, column_position]
big = df[df["mean radius"] > 20]
print(big.shape)

(569,)
(569, 2)
   mean radius  mean texture  mean perimeter  mean area
0        17.99         10.38          122.80     1001.0
1        20.57         17.77          132.90     1326.0
2        19.69         21.25          130.00     1203.0
3        11.42         20.38           77.58      386.1
(45, 31)


In [43]:
print(df["target"].value_counts()) # to check whether the dataset is balanced or skewed
print(df["target"].value_counts(normalize=True).round(3)) #gives percentage out of 100 of each class

target
1    357
0    212
Name: count, dtype: int64
target
1    0.627
0    0.373
Name: proportion, dtype: float64


In [45]:
print(df.groupby("target")["mean radius"].mean()) #made 2 groups based on output classes, 0 and 1. then found out mean for column mean radius for those two groups.

target
0    17.462830
1    12.146524
Name: mean radius, dtype: float64


In [51]:
X = df.drop(columns="target").to_numpy()
y = df["target"].to_numpy()
print("X shape: ", X.shape, "y shape: ", y.shape, "X dtype: ", X.dtype)

X shape:  (569, 30) y shape:  (569,) X dtype:  float64


In [ ]:
fig, axes = plt.subplots(1,3,figsize(15,4)) # fig mean canvas of size in inches, subplots (here 1 row, 3 columns) are actual graphs in the canvas.
axes[0].hist(df["mean radius"], bin=30) # bin is intervals
axes[0].set_title("Distribution of Mean Radius")
axes[0].set_xlabel("Mean Radius")
axes[0].set_ylabel("Count")

axes[1].scatter(df["mean radius"], df["mean texture"], c=df["target"], alpha=0.6) # x-axis, y-axis, c for color (different per class in "targer"), and alpha is transparency (as more points overlap transparency increases in the grpah).
axes[1].set_title("Radius vs texture, coloured by class")
axes[1].set_xlabel("mean radius")
axes[1].set_ylabel("mean textrue")

counts = df["target"].value_counts().sort_index()
axes[2].bar()